# Taylor-Window STFT Speech Denoiser

Removes stationary background noise (HVAC rumble, fan hiss, 50/60 Hz hum, traffic) from a human speech recording using a **Taylor-window** short-time Fourier transform (STFT) Analysis-Modification-Synthesis (AMS) pipeline.

Full mathematical spec and rationale: [`../docs/taylor_windows.md`](../docs/taylor_windows.md).

**Additive noise model:** the recorded signal is modeled as clean speech plus uncorrelated noise,
$$y[n] = s[n] + d[n]$$
and the goal is to recover an estimate $\hat{s}[n]$ of the clean speech.

**Why a Taylor window?** Standard tapers (Hann, Hamming, Blackman) couple mainlobe width and sidelobe attenuation together via one fixed shape. The Taylor window decouples them: you independently choose the peak sidelobe level ($SLL$, in dB) and the number of near-in equal-height sidelobes ($\bar{n}$), after which sidelobes decay monotonically at 6 dB/octave. This gives:

- **Deep, tunable sidelobe suppression** ($SLL=70$–$85$ dB) so low-frequency noise energy can't leak into the vocal formant region (100 Hz–3 kHz) and get mistaken for speech, or vice versa.
- **Monotonic decay, no edge discontinuity** (unlike Dolph-Chebyshev), which avoids Gibbs-phenomenon-driven "musical noise" artifacts between frames.
- **A sharp mainlobe** relative to Blackman/high-order Kaiser, preserving low-energy unvoiced sibilants (/s/, /f/, /t/, /k/) instead of smearing them into the noise estimate.

**Pipeline (8 stages implemented below):**

1. Frame the signal with 75% overlap ($R = N/4$) and apply the analysis Taylor window.
2. Forward real FFT (`scipy.fft.rfft`) per frame.
3. Split into magnitude and phase; phase is carried through unmodified (short-time phase errors are largely inaudible).
4. Track the noise floor per frequency bin with a **Minimum Statistics** estimator (recursive smoothing + sliding-minimum), no voice-activity detector needed.
5. Compute an adaptive **oversubtraction Wiener-style gain** per bin from the estimated SNR.
6. Apply the gain to the magnitude, recombine with the original phase.
7. Inverse real FFT, apply the synthesis Taylor window (same window as analysis).
8. Overlap-add (WOLA) into the output buffer, normalized by the running sum of squared window values.

We use `scipy.fft` / `scipy.signal.windows.taylor` throughout (not `numpy.fft`, not a precomputed window table) for full float64 precision and to allow dynamic re-parameterization per input sample rate.

In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Mapping, Optional

import numpy as np
import scipy.fft as spfft
import matplotlib.pyplot as plt
import soundfile as sf
import av
from scipy.signal.windows import taylor as taylor_window
from scipy.io import wavfile
from IPython.display import Audio, display

np.set_printoptions(precision=6, suppress=True)

We import `scipy.fft` rather than `numpy.fft`: it exposes a `workers=` argument for multi-threaded batched transforms, and its real-input `rfft`/`irfft` compute only the non-redundant half of the spectrum (`N//2+1` bins instead of `N`) — roughly half the FFT butterflies of a full complex `fft`/`ifft` for the same `N`, since our audio signal is strictly real-valued. `scipy.signal.windows.taylor` is called at runtime (not precomputed/hardcoded) so the window can be re-parameterized dynamically per the detected input sample rate.

## Configuration

Edit `INPUT_AUDIO_PATH` below to point at your own recording. Everything else has sensible defaults auto-selected from the sample rate (see the parameter table further down); leave the `OVERRIDE_*` fields as `None` unless you want to force specific values.

**All paths are anchored to `PROJECT_ROOT`**, found by walking up from the current working directory to the folder containing `pyproject.toml`. This matters: a Jupyter kernel's CWD is the *notebook's* directory, so a bare relative path like `"input/test.mp3"` resolves against `notebooks/` and fails with a confusing `No such file or directory`.

In [ ]:
def _find_project_root(start: Path | None = None) -> Path:
    """Walk up from `start` (default CWD) to the directory holding pyproject.toml.

    A Jupyter kernel's CWD is the *notebook's* directory, so a bare relative path
    like "input/test.mp3" resolves against `notebooks/`, not the repo root. Anchoring
    every path to PROJECT_ROOT makes the notebook run identically from either CWD.
    """
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    return start


PROJECT_ROOT = _find_project_root()

# --- Required: point this at your own recording ---
# (.wav, .flac, .mp3, .m4a, .mp4 all supported; the extension is not trusted --
#  the bundled test.mp3 is really an MP4/AAC container, which Stage 2 of the loader handles.)
INPUT_AUDIO_PATH = PROJECT_ROOT / "input" / "test.mp3"  # <-- EDIT THIS
OUTPUT_WAV_PATH = PROJECT_ROOT / "output_denoise" / "recording.wav"
OUTPUT_WAV_PATH.parent.mkdir(parents=True, exist_ok=True)

# --- STFT parameter overrides (None = auto-select from sample rate) ---
OVERRIDE_N: Optional[int] = None
OVERRIDE_R: Optional[int] = None
OVERRIDE_SLL_DB: Optional[float] = None
OVERRIDE_NBAR: Optional[int] = None

# --- Minimum-statistics noise tracking ---
GAMMA = 0.90              # recursive smoothing memory factor
MU = 1.35                 # noise over-estimation factor
NOISE_WINDOW_SECONDS = 1.5  # sliding minimum-tracking window duration

# --- Oversubtraction gain ---
BETA = 0.015              # spectral floor (~ -36.5 dB attenuation)
ALPHA_MIN, ALPHA_MAX = 1.0, 3.0   # oversubtraction factor range (high-SNR -> low-SNR)
SNR_LOW_DB, SNR_HIGH_DB = 0.0, 20.0  # alpha_m(SNR) interpolation anchors

# --- Misc ---
STORE_SPECTROGRAMS = True  # keep per-frame magnitude matrices for plotting (disable for very long inputs)
FFT_WORKERS = -1           # only used for the optional one-shot vectorized "before" spectrogram

# --- Visualization limits ---
# A 27-minute 16 kHz file is ~204k frames x 257 bins. Plotting that raw would build a
# ~52M-quad pcolormesh and base64-embed >100 MB of audio into the .ipynb, so the display
# cells decimate; the saved WAV and the transcript always cover the full recording.
WAVEFORM_MAX_POINTS = 4000      # min/max envelope columns for the before/after waveform plot
SPECTROGRAM_MAX_COLUMNS = 2000  # max-pooled time columns for the before/after spectrogram plot
AUDIO_PREVIEW_SECONDS = 30.0    # cap on the inline Audio() players (None = embed everything)

print(f"PROJECT_ROOT = {PROJECT_ROOT}  (cwd = {Path.cwd()})")

## Parameter selection

The Taylor window's non-zero edge pedestal means a standard 50% frame overlap leaves ~83% WOLA reconstruction ripple — we always use **75% overlap** ($R = N/4$), which drops ripple below 0.15%. Frame length, sidelobe level, and near-sidelobe count are chosen from the sample rate per this table (from `docs/taylor_windows.md`):

| Sample rate | Frame length $N$ | Hop $R$ (75%) | $SLL$ | $\bar{n}$ |
|---|---|---|---|---|
| 16 kHz | 512 (32.0 ms) | 128 | 70 dB | 5 |
| 44.1 kHz | 2048 (46.4 ms) | 512 | 80 dB | 6 |
| 48 kHz | 2048 (42.6 ms) | 512 | 85 dB | 6 |

For sample rates outside these anchors, `select_stft_params` clamps to the nearest anchor below 16 kHz / above 48 kHz, and linearly interpolates frame duration, $SLL$, and $\bar{n}$ between the two bracketing anchors otherwise.

In [ ]:
@dataclass(frozen=True)
class TaylorSTFTParams:
    fs: int
    N: int
    R: int
    sll_db: float
    nbar: int


@dataclass(frozen=True)
class DenoiseParams:
    gamma: float = 0.90
    mu: float = 1.35
    noise_window_seconds: float = 1.5
    beta: float = 0.015
    alpha_min: float = 1.0
    alpha_max: float = 3.0
    snr_low_db: float = 0.0
    snr_high_db: float = 20.0


# (fs, frame_duration_seconds, sll_db, nbar)
_ANCHORS = (
    (16_000, 0.0320, 70.0, 5),
    (44_100, 0.0464, 80.0, 6),
    (48_000, 0.0426, 85.0, 6),
)


def select_stft_params(fs: int, overrides: Optional[Mapping[str, Any]] = None) -> TaylorSTFTParams:
    """Pick (N, R, SLL, nbar) for a given sample rate per the Taylor-window spec table.

    Clamps to the nearest anchor outside [16 kHz, 48 kHz]; linearly interpolates
    frame duration / SLL / nbar between the two bracketing anchors otherwise.
    `overrides` may force specific fields (a dict subset of "N", "R", "sll_db", "nbar").
    """
    overrides = overrides or {}
    rates = [a[0] for a in _ANCHORS]

    if fs <= rates[0]:
        _, dur, sll, nbar = _ANCHORS[0]
    elif fs >= rates[-1]:
        _, dur, sll, nbar = _ANCHORS[-1]
    else:
        lo = max(a for a in _ANCHORS if a[0] <= fs)
        hi = min(a for a in _ANCHORS if a[0] >= fs)
        if lo[0] == hi[0]:
            _, dur, sll, nbar = lo
        else:
            t = (fs - lo[0]) / (hi[0] - lo[0])
            dur = lo[1] + t * (hi[1] - lo[1])
            sll = lo[2] + t * (hi[2] - lo[2])
            nbar = round(lo[3] + t * (hi[3] - lo[3]))

    N = overrides.get("N")
    if N is None:
        N = max(64, 2 * round(dur * fs / 2))  # round to nearest even length
    R = overrides.get("R", N // 4)  # 75% overlap by construction
    sll_db = overrides.get("sll_db", sll)
    nbar = overrides.get("nbar", int(nbar))
    return TaylorSTFTParams(fs=fs, N=int(N), R=int(R), sll_db=float(sll_db), nbar=int(nbar))

## Taylor window generation

Generated dynamically via `scipy.signal.windows.taylor`, peak-normalized to 1.0. The **same** window array is reused for both analysis (framing) and synthesis (before overlap-add) — this is required for the WOLA reconstruction formula below and must not be substituted with a different "dual" window by any higher-level abstraction.

In [ ]:
def make_taylor_window(N: int, nbar: int, sll_db: float) -> np.ndarray:
    """N-point symmetric Taylor window, peak-normalized to 1.0, float64."""
    w = taylor_window(N, nbar=nbar, sll=sll_db, norm=True, sym=True)
    return w.astype(np.float64)

## Load audio

Resilient multi-format loader (WAV, FLAC, MP3, M4A, MP4). Two-stage fallback per [`../docs/sound.md`](../docs/sound.md):

1. **`soundfile`** (libsndfile) — fast path for WAV, FLAC, and native MP3.
2. **PyAV** (`av`) — direct FFmpeg C-binding fallback for M4A/AAC/MP4 containers that libsndfile can't parse.

Both stages downmix to mono and normalize to float64 in $[-1, 1]$ regardless of source bit depth/channel count. The ffmpeg-subprocess Stage 3 from `docs/sound.md` is intentionally omitted here — no system `ffmpeg` binary is on PATH and `soundfile`+`PyAV` already cover every format this project needs; adding it would pull in `imageio-ffmpeg` purely as a corrupted-stream safety net.

**The file extension is not trusted.** The bundled `input/test.mp3` is not an MP3 at all — its header is `ftypmp42`, i.e. an **MP4 v2 container holding a mono 16 kHz AAC stream**. libsndfile cannot parse it, so Stage 1 *always* fails on this file and Stage 2 is what actually decodes it. This is precisely the case the fallback exists for, so a Stage-1 failure is normal and not an error condition.

Because of that, the loader records **every** stage's exception and reports them all if the file is genuinely unloadable. The earlier single-`last_exception` version discarded the informative libsndfile diagnostic and surfaced only the final stage's message, which made a simple wrong-CWD path bug look like a codec problem. A missing file is now also caught up front and reported with its fully resolved absolute path.

In [ ]:
class AudioLoadError(Exception):
    """Raised when every audio-loading stage fails."""


def _normalize_to_float32(data: np.ndarray) -> np.ndarray:
    """Scale integer/float PCM buffers to float32 in [-1, 1]."""
    if np.issubdtype(data.dtype, np.floating):
        return np.clip(data, -1.0, 1.0).astype(np.float32)
    if data.dtype == np.int16:
        return data.astype(np.float32) / 32768.0
    if data.dtype == np.int32:
        return data.astype(np.float32) / 2147483648.0
    if data.dtype == np.uint8:
        return (data.astype(np.float32) - 128.0) / 128.0
    raise TypeError(f"Unsupported input array dtype: {data.dtype}")


def describe_audio_container(path: str | Path) -> str:
    """Best-effort 'container/codec, N ch @ F Hz' probe. Never raises."""
    try:
        with av.open(str(path)) as container:
            stream = next(s for s in container.streams if s.type == "audio")
            ctx = stream.codec_context
            return (
                f"{container.format.name}/{ctx.name}, "
                f"{ctx.channels} ch @ {ctx.sample_rate} Hz, {ctx.format.name}"
            )
    except Exception as exc:  # probing is diagnostic only - never fail the load over it
        return f"<probe failed: {type(exc).__name__}: {exc}>"


def _load_audio_resilient_stereo(path: str | Path) -> tuple[np.ndarray, int]:
    """Load WAV/FLAC/MP3/M4A/MP4 into (channels, samples) float32 in [-1, 1].

    Stage 1: soundfile (libsndfile) - WAV, FLAC, native MP3.
    Stage 2: PyAV - M4A/AAC/MP4 containers soundfile can't parse.

    Every stage's exception is retained: on a file whose extension misreports its
    container (e.g. an MP4/AAC stream named .mp3) Stage 1 always fails and Stage 2
    succeeds, so keeping only the *last* error would hide the informative one.
    """
    p = Path(path)
    if not p.is_file():
        raise AudioLoadError(f"Audio file not found: {p.resolve()}")
    path_str = str(p)
    failures: list[tuple[str, Exception]] = []

    try:
        data, sr = sf.read(path_str, dtype="float32", always_2d=True)
        return data.T, int(sr)
    except Exception as e:
        failures.append(("soundfile/libsndfile", e))

    try:
        with av.open(path_str) as container:
            audio_streams = [s for s in container.streams if s.type == "audio"]
            if not audio_streams:
                raise AudioLoadError("No audio stream found in container.")
            stream = audio_streams[0]
            sr = stream.codec_context.sample_rate
            frames = []
            for frame in container.decode(stream):
                arr = _normalize_to_float32(frame.to_ndarray())
                if arr.ndim == 1:
                    arr = arr[np.newaxis, :]
                elif arr.ndim > 2:
                    arr = arr.reshape(-1, arr.shape[-1])
                frames.append(arr)
        if not frames:
            raise AudioLoadError("PyAV decoded zero audio frames.")
        return np.concatenate(frames, axis=1), int(sr)
    except Exception as e:
        failures.append(("PyAV/FFmpeg", e))

    detail = "\n".join(f"  - {stage}: {type(e).__name__}: {e}" for stage, e in failures)
    raise AudioLoadError(f"Failed to load audio file {p.resolve()}. All stages failed:\n{detail}")


def load_audio_mono_f64(path: str | Path) -> tuple[np.ndarray, int]:
    """Load WAV/FLAC/MP3/M4A/MP4, downmix to mono, return (signal, fs) as float64 in [-1, 1]."""
    signal, fs = _load_audio_resilient_stereo(path)
    if signal.shape[0] > 1:
        signal = signal.mean(axis=0, keepdims=True)
    x = np.clip(signal[0], -1.0, 1.0).astype(np.float64)
    return x, fs


x_noisy, fs = load_audio_mono_f64(INPUT_AUDIO_PATH)
stft_params = select_stft_params(
    fs,
    overrides={
        k: v
        for k, v in {
            "N": OVERRIDE_N,
            "R": OVERRIDE_R,
            "sll_db": OVERRIDE_SLL_DB,
            "nbar": OVERRIDE_NBAR,
        }.items()
        if v is not None
    },
)
denoise_params = DenoiseParams(
    gamma=GAMMA,
    mu=MU,
    noise_window_seconds=NOISE_WINDOW_SECONDS,
    beta=BETA,
    alpha_min=ALPHA_MIN,
    alpha_max=ALPHA_MAX,
    snr_low_db=SNR_LOW_DB,
    snr_high_db=SNR_HIGH_DB,
)
win = make_taylor_window(stft_params.N, stft_params.nbar, stft_params.sll_db)

print(f"Loaded  {INPUT_AUDIO_PATH}")
print(f"  actual container: {describe_audio_container(INPUT_AUDIO_PATH)}")
print(f"  decoded: fs={fs} Hz, {len(x_noisy)} samples ({len(x_noisy) / fs:.2f} s), dtype={x_noisy.dtype}")
print(f"  level:   rms={np.sqrt(np.mean(x_noisy ** 2)):.5f}  peak={np.abs(x_noisy).max():.4f}")
print(stft_params)

## Window shape and sidelobe attenuation

Sanity-check the generated Taylor window against a Hann window of the same length: time-domain shape, and the DTFT magnitude (heavily zero-padded for a fine-grained frequency axis) showing the sidelobe floor each achieves.

In [ ]:
def dtft_db(window: np.ndarray, zero_pad_factor: int = 16) -> tuple[np.ndarray, np.ndarray]:
    """Zero-padded rfft magnitude of `window`, normalized to 0 dB peak."""
    n_fft = window.size * zero_pad_factor
    spec = spfft.rfft(window, n=n_fft)
    mag_db = 20 * np.log10(np.maximum(np.abs(spec) / np.abs(spec).max(), 1e-12))
    freqs_norm = np.arange(mag_db.size) / n_fft  # cycles/sample, [0, 0.5]
    return freqs_norm, mag_db


hann_ref = np.hanning(stft_params.N)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(win, label=f"Taylor (SLL={stft_params.sll_db:.0f} dB, nbar={stft_params.nbar})")
axes[0].plot(hann_ref, label="Hann", alpha=0.7)
axes[0].set_title("Window shape (time domain)")
axes[0].set_xlabel("Sample")
axes[0].legend()

f_taylor, db_taylor = dtft_db(win)
f_hann, db_hann = dtft_db(hann_ref)
axes[1].plot(f_taylor, db_taylor, label="Taylor")
axes[1].plot(f_hann, db_hann, label="Hann", alpha=0.7)
axes[1].set_title("DTFT magnitude (sidelobe comparison)")
axes[1].set_xlabel("Normalized frequency (cycles/sample)")
axes[1].set_ylabel("dB")
axes[1].set_ylim(-140, 5)
axes[1].legend()

fig.tight_layout()
plt.show()

## Minimum Statistics noise estimator

Tracks a per-bin noise magnitude estimate without a voice-activity detector: a recursive first-order smoother produces $P_m(k) = \gamma P_{m-1}(k) + (1-\gamma) |X_m(k)|$, and the noise estimate is $\mu$ times the minimum of $P_m(k)$ over the trailing `NOISE_WINDOW_SECONDS`. Because the Taylor window keeps vocal formant energy from leaking into neighboring bins, this minimum tracker stays accurate even during active speech.

The trailing window is a preallocated `(L, num_bins)` ring buffer so the per-frame minimum is a single vectorized `ndarray.min(axis=0)` reduction, not a Python-level loop or a growing list.

In [ ]:
class MinStatNoiseEstimator:
    """Recursive-smoothing + sliding-minimum per-bin noise magnitude tracker."""

    def __init__(self, num_bins: int, L: int, gamma: float = 0.90, mu: float = 1.35):
        self.gamma = gamma
        self.mu = mu
        self.L = L
        self.P = np.zeros(num_bins, dtype=np.float64)
        self._ring = np.zeros((L, num_bins), dtype=np.float64)
        self._count = 0
        self._idx = 0

    def update(self, mag_frame: np.ndarray) -> np.ndarray:
        """Feed |X_m(k)| for the current frame; returns N_hat_m(k), shape (num_bins,)."""
        if self._count == 0:
            self.P[:] = mag_frame
        else:
            self.P = self.gamma * self.P + (1.0 - self.gamma) * mag_frame
        self._ring[self._idx] = self.P
        self._idx = (self._idx + 1) % self.L
        self._count = min(self._count + 1, self.L)
        window = self._ring if self._count == self.L else self._ring[: self._count]
        return self.mu * window.min(axis=0)

## Adaptive oversubtraction gain

$$G_m(k) = \max\left(\sqrt{1 - \alpha_m \left(\frac{\hat{N}_m(k)}{|X_m(k)|}\right)^2},\ \beta\right)$$

$\alpha_m$ is picked per frame from its segmental SNR using a Berouti-style piecewise-linear schedule: $\alpha_m = \alpha_{max}$ at or below `SNR_LOW_DB`, $\alpha_m = \alpha_{min}$ at or above `SNR_HIGH_DB`, linear in between. The term inside the square root is clipped to be non-negative *before* taking the square root — at low SNR with $\alpha_m$ up to 3.0, the (over-)estimated noise can exceed the frame magnitude at some bins, and an unclipped negative value would produce `NaN` that silently corrupts the entire reconstruction.

In [ ]:
def segmental_snr_db(mag_frame: np.ndarray, noise_mag: np.ndarray, eps: float = 1e-12) -> float:
    """Broadband frame SNR proxy: ratio of frame magnitude energy to estimated noise energy."""
    sig_e = float(np.sum(mag_frame**2))
    noise_e = float(np.sum(noise_mag**2))
    return 10.0 * math.log10((sig_e + eps) / (noise_e + eps))


def compute_alpha(snr_db: float, params: DenoiseParams) -> float:
    """Piecewise-linear oversubtraction schedule: alpha_max at snr_low_db, alpha_min at snr_high_db."""
    t = min(max((snr_db - params.snr_low_db) / (params.snr_high_db - params.snr_low_db), 0.0), 1.0)
    return params.alpha_max - (params.alpha_max - params.alpha_min) * t


def oversubtraction_gain(
    mag_frame: np.ndarray, noise_mag: np.ndarray, alpha: float, beta: float, eps: float = 1e-12
) -> np.ndarray:
    """G_m(k) = max(sqrt(clip(1 - alpha*(N_hat/|X|)^2, 0, None)), beta)."""
    ratio = noise_mag / np.maximum(mag_frame, eps)
    inner = np.clip(1.0 - alpha * ratio**2, 0.0, None)
    return np.maximum(np.sqrt(inner), beta)

## Main AMS pipeline

Ties together framing, forward FFT, noise tracking, gain computation, inverse FFT, and weighted overlap-add reconstruction. The frame loop is inherently sequential (the noise tracker's state depends on prior frames), so it is a plain Python loop over frames — each iteration does O(N log N) FFT work, which is fast enough for offline single-file processing.

Correctness details handled here:
- `rfft`/`irfft` produce/consume `N//2+1` bins, not `N` — all per-bin buffers are sized accordingly, and `irfft` is always called with an explicit `n=N`.
- The signal is zero-padded by `N-R` samples at the head and up to a full extra hop at the tail, so every original sample gets full window overlap and the output is trimmed back to exactly the input length.
- Every buffer is float64 end-to-end; only the final WAV write downcasts to int16.
- Overlap-add is normalized by the running sum of squared window values (`win_sq_sum`), clamped away from zero.

In [ ]:
@dataclass
class DenoiseResult:
    denoised: np.ndarray
    fs: int
    mag_before: Optional[np.ndarray] = None  # (num_bins, num_frames), for plotting only
    mag_after: Optional[np.ndarray] = None
    alpha_trace: np.ndarray = field(default_factory=lambda: np.empty(0))
    snr_trace: np.ndarray = field(default_factory=lambda: np.empty(0))


def denoise_signal(
    x: np.ndarray,
    stft_params: TaylorSTFTParams,
    denoise_params: DenoiseParams,
    win: np.ndarray,
    store_spectrograms: bool = True,
) -> DenoiseResult:
    N, R, fs_local = stft_params.N, stft_params.R, stft_params.fs
    num_bins = N // 2 + 1
    pad_left = N - R
    n_orig = len(x)

    # Pick num_frames so the last frame's end reaches at least `tail_needed`
    # (pad_left + n_orig), then pad the tail to exactly that total length —
    # guarantees every original sample is covered by at least one frame.
    tail_needed = pad_left + n_orig
    num_frames = max(1, math.ceil((tail_needed - N) / R) + 1)
    total_len = N + (num_frames - 1) * R
    n_pad_right = total_len - tail_needed

    padded = np.concatenate([np.zeros(pad_left), x, np.zeros(n_pad_right)]).astype(np.float64)

    L = math.ceil(denoise_params.noise_window_seconds * fs_local / R)
    noise_est = MinStatNoiseEstimator(num_bins, L, denoise_params.gamma, denoise_params.mu)

    out = np.zeros(len(padded), dtype=np.float64)
    win_sq_sum = np.zeros(len(padded), dtype=np.float64)
    mag_before = np.empty((num_bins, num_frames)) if store_spectrograms else None
    mag_after = np.empty((num_bins, num_frames)) if store_spectrograms else None
    alpha_trace = np.empty(num_frames)
    snr_trace = np.empty(num_frames)

    for m in range(num_frames):
        start = m * R
        frame = padded[start : start + N] * win
        spec = spfft.rfft(frame, n=N)
        mag = np.abs(spec)
        phase = np.angle(spec)

        noise_mag = noise_est.update(mag)
        snr_db = segmental_snr_db(mag, noise_mag)
        alpha = compute_alpha(snr_db, denoise_params)
        gain = oversubtraction_gain(mag, noise_mag, alpha, denoise_params.beta)

        mag_hat = gain * mag
        spec_hat = mag_hat * np.exp(1j * phase)
        synth = spfft.irfft(spec_hat, n=N) * win

        out[start : start + N] += synth
        win_sq_sum[start : start + N] += win**2

        if store_spectrograms:
            mag_before[:, m] = mag
            mag_after[:, m] = mag_hat
        alpha_trace[m] = alpha
        snr_trace[m] = snr_db

    out /= np.maximum(win_sq_sum, 1e-12)
    denoised = out[pad_left : pad_left + n_orig]
    return DenoiseResult(denoised, fs_local, mag_before, mag_after, alpha_trace, snr_trace)

In [ ]:
result = denoise_signal(x_noisy, stft_params, denoise_params, win, STORE_SPECTROGRAMS)
assert np.isfinite(result.denoised).all(), "denoised output contains NaN/Inf"
assert len(result.denoised) == len(x_noisy), "output length must match input length"
print(f"Denoised {len(result.denoised)} samples, all finite, length matches input.")

## Save denoised output

In [ ]:
def save_audio_int16(path: str | Path, x: np.ndarray, fs: int) -> None:
    """Clip to [-1, 1], scale to int16, and write a PCM WAV file."""
    x_clipped = np.clip(x, -1.0, 1.0)
    pcm16 = (x_clipped * 32767.0).astype(np.int16)
    wavfile.write(str(path), fs, pcm16)


save_audio_int16(OUTPUT_WAV_PATH, result.denoised, fs)
print(
    f"Wrote {OUTPUT_WAV_PATH}  "
    f"({OUTPUT_WAV_PATH.stat().st_size / 1e6:.1f} MB, "
    f"{len(result.denoised)} samples @ {fs} Hz = {len(result.denoised) / fs:.1f} s)"
)

## Visualize: waveform before / after

In [ ]:
def envelope_decimate(x: np.ndarray, fs: int, max_points: int) -> tuple[np.ndarray, np.ndarray]:
    """Return (times, values) tracing the per-block min AND max of `x`.

    Plotting both extremes of each block (rather than striding, which would sample
    one arbitrary point per block) keeps peak excursion honest at high decimation
    ratios -- a 27-minute 16 kHz file is 26M points, ~6500x more than a plot can show.
    """
    n = x.size
    if n <= 2 * max_points:
        return np.arange(n) / fs, x
    block = math.ceil(n / max_points)
    trimmed = x[: (n // block) * block].reshape(-1, block)
    times = np.repeat(np.arange(trimmed.shape[0]) * block, 2) / fs
    values = np.empty(times.size)
    values[0::2] = trimmed.min(axis=1)
    values[1::2] = trimmed.max(axis=1)
    return times, values


t_noisy, env_noisy = envelope_decimate(x_noisy, fs, WAVEFORM_MAX_POINTS)
t_clean, env_clean = envelope_decimate(result.denoised, fs, WAVEFORM_MAX_POINTS)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)
axes[0].plot(t_noisy, env_noisy, linewidth=0.5)
axes[0].set_title("Noisy input")
axes[0].set_ylabel("Amplitude")
axes[1].plot(t_clean, env_clean, linewidth=0.5, color="tab:orange")
axes[1].set_title("Denoised output")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Amplitude")
fig.tight_layout()
plt.show()

if env_noisy.size < x_noisy.size:
    print(
        f"Min/max envelope: {x_noisy.size} samples -> {env_noisy.size} plotted points "
        f"({x_noisy.size / env_noisy.size:.0f}:1 decimation, "
        f"{math.ceil(x_noisy.size / WAVEFORM_MAX_POINTS) / fs * 1000:.0f} ms per block). "
        "Raise WAVEFORM_MAX_POINTS for finer detail."
    )

## Visualize: spectrogram before / after

Built directly from the pipeline's own stored magnitude matrices (`STORE_SPECTROGRAMS=True`), so the plot is faithful to what the algorithm actually computed rather than an independently-windowed spectrogram.

Long inputs are **max-pooled along the time axis** to at most `SPECTROGRAM_MAX_COLUMNS` columns first. At 16 kHz a 27-minute file is ~204k frames; handing that to `pcolormesh` raw would build a ~52M-quad mesh and exhaust memory. Pooling takes the **maximum** of each block rather than the mean — averaging ~100 frames into one washes out exactly the speech transients the plot exists to show. Both panels use the same block size and the same `vmin`/`vmax`, so before/after stay directly comparable.

In [ ]:
def pool_time(mag: np.ndarray, times: np.ndarray, block: int) -> tuple[np.ndarray, np.ndarray]:
    """Max-pool a (bins, frames) magnitude matrix by `block` frames along the time axis."""
    if block <= 1:
        return mag, times
    keep = (mag.shape[1] // block) * block
    pooled = mag[:, :keep].reshape(mag.shape[0], -1, block).max(axis=2)
    return pooled, times[:keep:block]


if not STORE_SPECTROGRAMS:
    print("STORE_SPECTROGRAMS is False — skipping spectrogram plots.")
elif result.mag_before is None:
    print("Spectrogram matrices were already released (see the metrics cell) — re-run the pipeline to plot.")
else:
    freqs = spfft.rfftfreq(stft_params.N, d=1.0 / fs)
    num_frames = result.mag_before.shape[1]
    frame_times = np.arange(num_frames) * stft_params.R / fs
    block = max(1, math.ceil(num_frames / SPECTROGRAM_MAX_COLUMNS))
    eps = 1e-12

    # Pool before taking the log, and free each ~0.4 GB intermediate immediately.
    pooled_before, pooled_times = pool_time(result.mag_before, frame_times, block)
    db_before = 20 * np.log10(np.maximum(pooled_before, eps))
    del pooled_before
    pooled_after, _ = pool_time(result.mag_after, frame_times, block)
    db_after = 20 * np.log10(np.maximum(pooled_after, eps))
    del pooled_after

    vmax = db_before.max()
    vmin = vmax - 100

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True, sharey=True)
    for ax, db, title in ((axes[0], db_before, "Noisy spectrogram"), (axes[1], db_after, "Denoised spectrogram")):
        mesh = ax.pcolormesh(pooled_times, freqs, db, shading="auto", vmin=vmin, vmax=vmax, cmap="magma")
        ax.set_title(title)
        ax.set_ylabel("Frequency (Hz)")
        fig.colorbar(mesh, ax=ax, label="dB")
    axes[1].set_xlabel("Time (s)")
    fig.tight_layout()
    plt.show()

    if block > 1:
        print(
            f"Max-pooled time axis: {num_frames} frames -> {db_before.shape[1]} columns "
            f"({block} frames = {block * stft_params.R / fs * 1000:.0f} ms per column). "
            "Raise SPECTROGRAM_MAX_COLUMNS for finer detail."
        )
    del db_before, db_after

## Metrics summary

In [ ]:
print(f"Segmental SNR (dB):   mean={result.snr_trace.mean():.2f}  median={np.median(result.snr_trace):.2f}")
print(f"Oversubtraction alpha: mean={result.alpha_trace.mean():.2f}  min={result.alpha_trace.min():.2f}  max={result.alpha_trace.max():.2f}")

if STORE_SPECTROGRAMS and result.mag_before is not None:
    # Noise-floor estimate: mean magnitude over the lowest-SNR decile of frames, before vs. after.
    low_snr_cutoff = np.percentile(result.snr_trace, 10)
    low_snr_frames = result.snr_trace <= low_snr_cutoff
    floor_before_db = 20 * np.log10(result.mag_before[:, low_snr_frames].mean() + 1e-12)
    floor_after_db = 20 * np.log10(result.mag_after[:, low_snr_frames].mean() + 1e-12)
    print(f"Noise-floor estimate (lowest-SNR decile of frames): before={floor_before_db:.1f} dB, after={floor_after_db:.1f} dB")
    print(f"Estimated noise-floor reduction: {floor_before_db - floor_after_db:.1f} dB")

    # Last use of the magnitude matrices. On a 27-minute 16 kHz file they hold ~0.84 GB,
    # which we want back before the Whisper model (~1.6 GB int8 on CPU) loads below.
    freed_gb = (result.mag_before.nbytes + result.mag_after.nbytes) / 1e9
    result.mag_before = None
    result.mag_after = None
    print(f"Released stored spectrogram matrices ({freed_gb:.2f} GB) ahead of the ASR stage.")

## Optional: sidelobe attenuation validation

Direct numeric check of the generated window against its configured `sll_db`, independent of the earlier comparison plot.

In [ ]:
def measured_peak_sidelobe_db(window: np.ndarray, zero_pad_factor: int = 32) -> float:
    """Peak sidelobe level: max dB magnitude past the first mainlobe null."""
    _, mag_db = dtft_db(window, zero_pad_factor)
    first_null = np.argmax(np.diff(mag_db) > 0)  # first local minimum -> end of mainlobe
    return float(mag_db[first_null:].max())


measured_sll = measured_peak_sidelobe_db(win)
print(f"Configured SLL: {stft_params.sll_db:.1f} dB   Measured peak sidelobe (worst-case, any sidelobe): {-measured_sll:.1f} dB")
if abs(measured_sll + stft_params.sll_db) > 3.0:
    print(
        "Note: measured worst-case sidelobe differs from the configured SLL by more than 3 dB. "
        "This is expected for some (SLL, nbar) combinations — the SLL guarantee strictly covers "
        "only the first nbar-1 near-in sidelobes; sidelobes in the transition region before the "
        "asymptotic 6 dB/octave rolloff can transiently exceed it. Not a bug in the window generation."
    )

## Listen

Only renders audio players in a live Jupyter/IPython session.

`IPython.display.Audio` base64-embeds the entire array **into the `.ipynb` file** — for a 27-minute 16 kHz recording that is ~70 MB of text per player. The players below are therefore capped to the first `AUDIO_PREVIEW_SECONDS`; set that knob to `None` to embed everything. The complete denoised recording is always on disk at `OUTPUT_WAV_PATH`.

In [ ]:
def preview_slice(x: np.ndarray, fs: int, seconds: Optional[float]) -> tuple[np.ndarray, bool]:
    """Head slice of `x` for inline playback; returns (slice, was_truncated)."""
    if seconds is None:
        return x, False
    n = int(seconds * fs)
    if x.size <= n:
        return x, False
    return x[:n], True


noisy_preview, truncated = preview_slice(x_noisy, fs, AUDIO_PREVIEW_SECONDS)
clean_preview, _ = preview_slice(result.denoised, fs, AUDIO_PREVIEW_SECONDS)

if truncated:
    print(
        f"Previewing the first {noisy_preview.size / fs:.0f} s of {x_noisy.size / fs:.0f} s "
        f"(AUDIO_PREVIEW_SECONDS). Full denoised audio: {OUTPUT_WAV_PATH}"
    )

print("Noisy input:")
display(Audio(noisy_preview, rate=fs))
print("Denoised output:")
display(Audio(clean_preview, rate=fs))

## Speech-to-Text Transcription (Thai + English)

Transcribes the denoised output (`OUTPUT_WAV_PATH`) with [`faster-whisper`](https://github.com/SYSTRAN/faster-whisper), a CTranslate2 reimplementation of Whisper (fused C++/CUDA kernels, int8/float16 quantization). Full rationale: [`../docs/whisper.md`](../docs/whisper.md).

**Device selection is automatic**: probes for a usable CUDA GPU via `ctranslate2.get_cuda_device_count()` and falls back to CPU (with a plain try/except safety net in case a CUDA device is present but unusable) — no manual `device=` editing needed.

**Thai + English code-switching**: `ASR_LANGUAGE="th"` anchors decoding to Thai (best accuracy for Thai phonemes/tone marks) while still transcribing inline English loanwords as-is — Whisper's language setting conditions decoding, it doesn't hard-filter the vocabulary. An `initial_prompt` reinforces common Thai-context English terms (Application, Cloud, Zoom, etc.).

In [ ]:
import os
import time

import ctranslate2
from faster_whisper import WhisperModel, BatchedInferencePipeline

In [ ]:
# --- Model + device ---
ASR_MODEL_SIZE = "large-v3-turbo"   # best Thai/EN code-switch accuracy per docs/whisper.md; already the low-VRAM/fast variant
ASR_LANGUAGE = "th"                 # anchor language; set to None to auto-detect instead
# None = auto (min(4, os.cpu_count())). This box is CPU-only with 16 cores, and a 27-minute
# file at 4 threads is needlessly slow, so pin higher. CTranslate2 honours cpu_threads directly.
ASR_CPU_THREADS: Optional[int] = 8

# --- GPU batching heuristic ---
# BatchedInferencePipeline gives a real throughput win on longer files (VAD-sliced segments run
# concurrently through the encoder/decoder), but for short clips the batch-assembly overhead can
# outweigh the benefit and it bypasses some hallucination guardrails (see docs/whisper.md) — so we
# only use it on GPU, and only past a minimum duration.
ASR_USE_BATCHED_ON_GPU = True
ASR_BATCH_SIZE = 16
ASR_BATCH_MIN_AUDIO_SECONDS = 60.0

# --- VAD tuning (Thai: catches short particles like "ครับ"/"ค่ะ" and preserves tone-mark tails) ---
ASR_VAD_PARAMETERS = {
    "threshold": 0.30,
    "min_speech_duration_ms": 200,
    "max_speech_duration_s": 12,
    "min_silence_duration_ms": 400,
    "speech_pad_ms": 400,
}

# --- Decode / anti-hallucination hyperparameters ---
ASR_BEAM_SIZE = 5
ASR_PATIENCE = 1.2
ASR_BEST_OF = 5
ASR_LENGTH_PENALTY = 1.0
ASR_REPETITION_PENALTY = 1.25
ASR_NO_REPEAT_NGRAM_SIZE = 4
ASR_TEMPERATURE = [0.0, 0.2, 0.4, 0.6]
ASR_COMPRESSION_RATIO_THRESHOLD = 2.2
ASR_LOG_PROB_THRESHOLD = -0.9
ASR_NO_SPEECH_THRESHOLD = 0.5
ASR_INITIAL_PROMPT = (
    "ต่อไปนี้เป็นการบันทึกเสียงภาษาไทย มีการใช้คำทับศัพท์ภาษาอังกฤษ "
    "เช่น Application, AI, Cloud, Software, Meeting, Project, OK, Zoom, Facebook, Line"
)

# --- Output ---
ASR_TRANSCRIPT_OUTPUT_PATH = OUTPUT_WAV_PATH.with_suffix(".txt")

In [ ]:
def detect_device_and_compute_type(cpu_threads: Optional[int] = None) -> tuple[str, str, int]:
    """Pick (device, compute_type, cpu_threads) automatically.

    CUDA GPU present -> ("cuda", "int8_float16", threads); otherwise -> ("cpu", "int8", threads).
    """
    try:
        gpu_count = ctranslate2.get_cuda_device_count()
    except Exception:
        gpu_count = 0
    threads = cpu_threads if cpu_threads is not None else min(4, os.cpu_count() or 4)
    if gpu_count > 0:
        return "cuda", "int8_float16", threads
    return "cpu", "int8", threads

In [ ]:
def load_whisper_model(model_size: str, cpu_threads: Optional[int] = None) -> tuple[WhisperModel, str]:
    """Load a WhisperModel on the auto-detected device, falling back to CPU if CUDA init fails."""
    device, compute_type, threads = detect_device_and_compute_type(cpu_threads)
    os.environ["OMP_NUM_THREADS"] = str(threads)
    try:
        model = WhisperModel(model_size, device=device, compute_type=compute_type, cpu_threads=threads)
    except Exception as exc:
        if device != "cuda":
            raise
        print(f"CUDA model init failed ({exc}); falling back to CPU.")
        device, compute_type = "cpu", "int8"
        model = WhisperModel(model_size, device=device, compute_type=compute_type, cpu_threads=threads)
    print(f"Loaded Whisper model {model_size!r} on {device} (compute_type={compute_type}, cpu_threads={threads})")
    return model, device

Batching note: on CPU we always use `model.transcribe()` directly — the CPU profile in `docs/whisper.md` favors direct execution so the strict anti-hallucination thresholds (`compression_ratio_threshold`, `log_prob_threshold`) stay active, which `BatchedInferencePipeline` bypasses. On GPU, we only switch to `BatchedInferencePipeline` once the input is at least `ASR_BATCH_MIN_AUDIO_SECONDS` long — below that, VAD-segment batch assembly overhead outweighs the parallel-decode win.

In [ ]:
model, asr_device = load_whisper_model(ASR_MODEL_SIZE, ASR_CPU_THREADS)

transcribe_kwargs = dict(
    language=ASR_LANGUAGE,
    beam_size=ASR_BEAM_SIZE,
    patience=ASR_PATIENCE,
    best_of=ASR_BEST_OF,
    length_penalty=ASR_LENGTH_PENALTY,
    repetition_penalty=ASR_REPETITION_PENALTY,
    no_repeat_ngram_size=ASR_NO_REPEAT_NGRAM_SIZE,
    temperature=ASR_TEMPERATURE,
    initial_prompt=ASR_INITIAL_PROMPT,
    condition_on_previous_text=False,
    compression_ratio_threshold=ASR_COMPRESSION_RATIO_THRESHOLD,
    log_prob_threshold=ASR_LOG_PROB_THRESHOLD,
    no_speech_threshold=ASR_NO_SPEECH_THRESHOLD,
    suppress_blank=True,
    suppress_tokens=[-1],
    vad_filter=True,
    vad_parameters=ASR_VAD_PARAMETERS,
)

audio_duration_s = len(result.denoised) / fs
use_batched = (
    asr_device == "cuda"
    and ASR_USE_BATCHED_ON_GPU
    and audio_duration_s >= ASR_BATCH_MIN_AUDIO_SECONDS
)

print(f"Transcribing via {'BatchedInferencePipeline' if use_batched else 'WhisperModel.transcribe'} on {asr_device}")
print(f"Source: {OUTPUT_WAV_PATH} ({audio_duration_s / 60:.1f} min)")

t_start = time.perf_counter()
if use_batched:
    pipeline = BatchedInferencePipeline(model=model)
    segments_gen, info = pipeline.transcribe(str(OUTPUT_WAV_PATH), batch_size=ASR_BATCH_SIZE, **transcribe_kwargs)
else:
    segments_gen, info = model.transcribe(str(OUTPUT_WAV_PATH), **transcribe_kwargs)

# faster-whisper's `segments` is a lazy generator - the decode actually happens here.
segments = [{"start": seg.start, "end": seg.end, "text": seg.text} for seg in segments_gen]
elapsed = time.perf_counter() - t_start

transcript = " ".join(seg["text"].strip() for seg in segments)

print(f"Detected language: {info.language} (probability={info.language_probability:.2f})")
print(
    f"Decoded {len(segments)} segments in {elapsed / 60:.1f} min "
    f"({audio_duration_s / max(elapsed, 1e-9):.1f}x realtime)"
)
for seg in segments:
    print(f"[{seg['start']:6.2f}s -> {seg['end']:6.2f}s] {seg['text']}")

In [ ]:
ASR_TRANSCRIPT_OUTPUT_PATH.write_text(transcript, encoding="utf-8")
print(f"Wrote transcript ({len(segments)} segments, {len(transcript)} chars) to {ASR_TRANSCRIPT_OUTPUT_PATH}")
print("\nFull transcript:\n")
print(transcript)